In [ ]:
%load_ext autoreload
%autoreload 2

from sindy_wm.envs.episode import play_episode
from sindy_wm.envs.policies import AttackNearestPolicy
from sindy_wm.envs.scenarios import make_env
from sindy_wm.states.aggregate import level_a

In [ ]:
import pandas as pd

env = make_env("5m_vs_5m")
results = []
for aggression in [0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    for seed in range(10):
        episode = play_episode(env, AttackNearestPolicy(aggression), seed=seed)
        states = level_a(episode.snapshots, normalize=True)
        results.append(
            {
                "aggression": aggression,
                "seed": seed,
                "won": episode.battle_won,
                "steps": episode.n_steps,
                "red_health_lost": 1 - states["health_red"].iloc[-1],
                "states": states,
            }
        )

results = pd.DataFrame(results)
results.groupby("aggression")[["won", "steps", "red_health_lost"]].mean()

In [ ]:
from sindy_wm.evaluation.plots import plot_many_episodes

for aggression, group in results.groupby("aggression"):
    plot_many_episodes(group["states"].tolist(), title=f"5m_vs_5m, aggression {aggression}")

In [ ]:
plot_many_episodes(results["states"].tolist(), title="5m_vs_5m, all aggression values", alpha=0.15)